# Model Inverses
<!--
2026.06.12 DAJones: created this from "7D Inverse.ipyny"
2026.06.15 DAJones: merged in the even pattern and renamed the file
2026.08.23 DAJones: paying this file attention as the most important concept file.
-->
The even dimensions are simplest.
The odd dimensions require composite scalar/pseudoscalar processing: both in the tweak, 
and in the "determinant".
Odd cases durive FLS processing from the preceeding even case.
Think of the odd cases as extending the preceeding even case.

The last step to completing the general pattern was finding the right involutions.
Reversion sits at the base of every case above 6D but an additional involution is composed on top of the reversion.
Those reversion modifications are described below, and they follow a predictable pattern.
 - When we step from an odd dimension to an even one, we simply double the size of each piece of the odd base.
 - When we step from an even dimension to an odd one, we concatenate the complement of the even base.
The odd contribution is termed "Thue-Morse".  This is the pattern found in the pseudoscalar sign table.


In [1]:
import numpy as np
import clifford.util as util
import clifford.context as Clif
from clifford.multivector import Accum


In [ ]:
# Lounesto names for common involutions
#reversion = [[1,1,-1,-1][3 & Clif.Grade[i]] for i in range(1<<dimensions)]
#conjugation = [[1,-1,-1,1][3 & Clif.Grade[i]] for i in range(1<<dimensions)]
#grade_involution = [[1,-1][1 & Clif.Grade[i]] for i in range(1<<dimensions)]

invo_6   = [ 1]*64
invo_7   = [ 1]*64  + [-1]*64
invo_8   = [ 1]*128 + [-1]*128
invo_9   = [ 1]*128 + [-1]*128 + [-1]*128 + [ 1]*128
invo_10  = [ 1]*256 + [-1]*256 + [-1]*256 + [ 1]*256
invo_11  = [ 1]*256 + [-1]*256 + [-1]*256 + [ 1]*256 + [-1]*256 + [ 1]*256 + [ 1]*256 + [-1]*256
invo_12  = [ 1]*512 + [-1]*512 + [-1]*512 + [ 1]*512 + [-1]*512 + [ 1]*512 + [ 1]*512 + [-1]*512
invo_13  = [ 1]*512 + [-1]*512 + [-1]*512 + [ 1]*512 + [-1]*512 + [ 1]*512 + [ 1]*512 + [-1]*512
invo_13 += [-1]*512 + [ 1]*512 + [ 1]*512 + [-1]*512 + [ 1]*512 + [-1]*512 + [-1]*512 + [ 1]*512

involution_fix = (invo_6, invo_7, invo_8, invo_9, invo_10, invo_11, invo_12, invo_13)

reverse_pattern = [1,1,-1,-1]
def inverse_involution(dimensions):
    if dimensions < 6 or dimensions > 13:
        print("use another method")
        return None
    fix = involution_fix[dimensions-6]
    return [[1,1,-1,-1][3 & Clif.Grade[n]] * fix[n] for n in range(1<<dimensions)]

tweak_6_7   = [3, 1, 1/3]
tweak_8_9   = [7, 3, 5/3, 1, 3/5, 1/3, 1/7]
tweak_10_11 = [15, 7, 13/3, 3, 11/5, 5/3, 9/7, 1, 7/9, 3/5, 5/11, 1/3, 3/13, 1/7, 1/15]
tweak_12_13 = [31, 15, 29/3, 7, 27/5, 13/3, 25/7, 3, 23/9, 11/5, 21/11, 5/3, 19/13, 9/7, 17/15, 1,
             15/17, 7/9, 13/19, 3/5, 11/21, 5/11, 9/23, 1/3, 7/25, 3/13, 5/27, 1/7, 3/29, 1/15, 1/31]
#15/1,14/2,13/3,12/4,11/5,10/6,9/7,8/8,7/9,6/10,5/11,4/12,3/13,2/14,1/15
#N=1<<D;[(N-n)/n for n in range(1,N)]
tweak = [        #     FLS  N      
    None,        #  0
    None,        #  1
    None,        #  2
    None,        #  3  
    None,        #  4
    None,        #  5
    tweak_6_7,   #  6   3   4
    tweak_6_7,   #  7   4   4
    tweak_8_9,   #  8   5   8
    tweak_8_9,   #  9   6   8
    tweak_10_11, # 10   7  16
    tweak_10_11, # 11   8  16
    tweak_12_13, # 12   9  32
    tweak_12_13, # 13  10  32
]

In [11]:
# the general form of the tweak array is
def make_tweak(n):
    m = 1<<n
    t = []
    for k in range(1,m):
        t += [f"{m-k}/{k}"]
    return t
for i in range(1,6):
    print(i,make_tweak(i))

1 ['1/1']
2 ['3/1', '2/2', '1/3']
3 ['7/1', '6/2', '5/3', '4/4', '3/5', '2/6', '1/7']
4 ['15/1', '14/2', '13/3', '12/4', '11/5', '10/6', '9/7', '8/8', '7/9', '6/10', '5/11', '4/12', '3/13', '2/14', '1/15']
5 ['31/1', '30/2', '29/3', '28/4', '27/5', '26/6', '25/7', '24/8', '23/9', '22/10', '21/11', '20/12', '19/13', '18/14', '17/15', '16/16', '15/17', '14/18', '13/19', '12/20', '11/21', '10/22', '9/23', '8/24', '7/25', '6/26', '5/27', '4/28', '3/29', '2/30', '1/31']


In [3]:
VERBOSE = True
def even_inverse( T, I, A ): # T = tweaks, I = involution, A = multivector
    B = I * A
    C = A * B # C = AB -> 1/A = B/C
    D = C.copy()
    D.Reg[0] *= -T[0]
    for i in range(1,len(T)):
        D *= C
        D.Reg[0] *= -T[i]
    E = D * C # scalar: E = DC -> 1/C = D/E -> 1/A = BD/E
    rC = (1/E.Reg[0]) * D
    rA = B * rC
    if VERBOSE:
        print('\nValidate that we have found rC = 1/C, by printing C * rC' )
        print( C * rC )
        print('\nValidate that we have found rA = 1/A, by printing A * rA' )
        print( A * rA )
    return rA


In [4]:
Dimensions = [6, 8, 10, 12]
for dimensions in Dimensions:
    print(f"Dimension = {dimensions}")
    Clif.Cl(dimensions)
    A = Accum()
    A.random()
    I = even_inverse( tweak[dimensions], inverse_involution(dimensions), A )


Dimension = 6
Initialize: len(Grade) =  64

Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000


Validate that we have found rA = 1/A, by printing A * rA
00000000       1.00000000

Dimension = 8
Initialize: len(Grade) =  256

Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000


Validate that we have found rA = 1/A, by printing A * rA
00000000       1.00000000

Dimension = 10
Initialize: len(Grade) =  1024

Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000


Validate that we have found rA = 1/A, by printing A * rA
00000000       1.00000000

Dimension = 12
Initialize: len(Grade) =  4096

Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000
00000001       0.00019728
00000010       0.00031016
00000011      -0.00002211
00000100       0.00003681
00000101      -0.00017064
00000110      -0.00018947
00000111      -0.00000376
00001000       0.00040915
00001001      -0.00

In [12]:
# This is the "tweak version" of the generalized 7D inverse copied from "A&S 7D Inverse.ipynb" in /Clifford/.
# This is a model written for clarity of concept without regard to efficiency.

VERBOSE = True

# T is the tweak list
# I is {-1,1} len(I)==len(A) describing the involution needed
def odd_inverse( T, I, A ):
    PSEUDO = (1<<A.dimensions)-1
    B = I * A
    C = A * B # C = AB -> 1/A = B/C
    D = C.copy()
    D.Reg[0] *= -T[0]
    D.Reg[PSEUDO] *= -T[0]
    for i in range(1,len(T)):
        D *= C
        D.Reg[0] *= -T[i]
        D.Reg[PSEUDO] *= -T[i]
    E = D * C # E[0,PSEUDO] are only non-zero elements
    # E=DC -> 1/C = D/E = E'D/E'E
    Ep = E.copy()
    Ep.Reg[PSEUDO] *= -1
    EpE = Ep * E # scalar only
    rC = (1 / EpE.Reg[0]) * Ep * D
    rA = B * rC # 1/A = B/C
    if VERBOSE:
        print('Validate that we have found rC = 1/C, by printing C * rC' )
        print( C * rC )
        print('Validate that we have found rA = 1/A, by printing A * rA' )
        print( A * rA )
    return rA


In [13]:
Dimensions = [7, 9, 11, 13]
for dimension in Dimensions:
    print(f"Dimension = {dimension}")
    Clif.Cl(dimension)
    A = Accum()
    A.random()
    I = odd_inverse( tweak[dimension], inverse_involution(dimension), A )


Dimension = 7
Initialize: len(Grade) =  128
Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000

Validate that we have found rA = 1/A, by printing A * rA
00000000       1.00000000

Dimension = 9
Initialize: len(Grade) =  512
Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000

Validate that we have found rA = 1/A, by printing A * rA
00000000       1.00000000

Dimension = 11
Initialize: len(Grade) =  2048
Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000

Validate that we have found rA = 1/A, by printing A * rA
00000000       1.00000000

Dimension = 13
Initialize: len(Grade) =  8192
Validate that we have found rC = 1/C, by printing C * rC
00000000       1.00000000
00000001      -0.00016886
00000010       0.00011701
00000011      -0.00061383
00000100      -0.00000421
00000101      -0.00060641
00000110       0.00049370
00000111       0.00014790
00001000      -0.00015629
00001001       0.00131251